# AI Investment Risk Radar: Exploratory Data Analysis & Research

This research notebook explores causal ETF indicators, downside target distributions, non-overlapping event rates, chronological split integrity, and proxy limitations for the **AI Investment Risk Radar** project.

> [!IMPORTANT]
> **Evaluation Integrity:**
> All exploratory analyses and parameter evaluations in this notebook are strictly confined to the **Train** and **Validation** partitions. The final holdout test partition (24 months) is never inspected or used for model tuning.


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from ai_risk_radar.config import load_settings
from ai_risk_radar.data import load_snapshot
from ai_risk_radar.features import build_features
from ai_risk_radar.splits import make_splits

settings = load_settings(root_dir / "config.json")
print(f"Loaded settings: tickers={settings.tickers}, horizon={settings.target_horizon}d, threshold={settings.target_threshold}")


## 1. Data Ingestion & Snapshot Verification

The project universe consists of four liquid US ETFs:
- `SOXX`: iShares Semiconductor ETF (hardware, AI acceleration proxy)
- `XLK`: Technology Select Sector SPDR Fund (enterprise software & big tech proxy)
- `SPY`: SPDR S&P 500 ETF Trust (broad equity market anchor)
- `SHY`: iShares 1-3 Year Treasury Bond ETF (defensive short-duration cash buffer)


In [ ]:
snapshot_dir = root_dir / settings.raw_dir / "snapshot"
bundle = load_snapshot(snapshot_dir)

print(f"Dataset sessions: {len(bundle.sessions)}")
print(f"Start date: {bundle.sessions[0].strftime('%Y-%m-%d')}")
print(f"End date: {bundle.sessions[-1].strftime('%Y-%m-%d')}")
print(f"Cutoff session: {bundle.manifest.get('cutoff_session')}")
print(f"SHA-256 (adjusted_close): {bundle.manifest['hashes']['adjusted_close.parquet']}")


## 2. Causal Feature Engineering & Downside Target

The target is defined as:
$$\text{Target}_t = \mathbb{I}\left( R_{t+1 \to t+20}^{\text{basket}} \le -0.05 \right)$$
where the reference basket is a 50/50 equal-weighted allocation to `SOXX` and `XLK`.


In [ ]:
features = build_features(bundle, settings)
df = features.frame
print(f"Features DataFrame shape: {df.shape}")
print(f"Feature columns: {list(features.feature_columns)}")
print(f"Target column: {features.target_columns[0]}")


## 3. Purged Chronological Splits & Label Boundaries

To prevent temporal leakage from overlapping 20-trading-day labels, a 20-session purge window is enforced at each partition boundary:
- **Train:** 2016-01-04 to 2021-11-18 (5.87 years)
- **Calibration:** 2021-12-17 to 2022-12-16 (12 months)
- **Validation:** 2023-01-18 to 2024-11-15 (24 months)
- **Holdout Test:** 2024-11-18 to 2026-11-16 (500 trading sessions, untouched)


In [ ]:
splits = make_splits(features, settings)
target_col = features.target_columns[0]

train_target = df.loc[splits.train, target_col]
cal_target = df.loc[splits.calibration, target_col]
val_target = df.loc[splits.validation, target_col]

print(f"Train samples: {len(splits.train)}, Downside Prevalence: {train_target.mean():.1%}")
print(f"Calibration samples: {len(splits.calibration)}, Downside Prevalence: {cal_target.mean():.1%}")
print(f"Validation samples: {len(splits.validation)}, Downside Prevalence: {val_target.mean():.1%}")


## 4. Non-Overlapping Event Distribution & Correlation Analysis

Because consecutive daily forward-looking 20-day returns overlap heavily, we analyze non-overlapping 20-session blocks to measure true independent downside regimes.


In [ ]:
# Inspect non-overlapping 20-session blocks in train
train_df = df.loc[splits.train]
non_ov_train = train_df.iloc[::20]
print(f"Non-overlapping 20-session blocks in Train: {len(non_ov_train)}")
print(f"Non-overlapping downside events: {int(non_ov_train[target_col].sum())} ({non_ov_train[target_col].mean():.1%})")

# Feature correlation with downside target in training split
corr = train_df[list(features.feature_columns) + [target_col]].corr()
print("\nCorrelations with Target:")
print(corr[target_col].sort_values(ascending=False))


## 5. Proxy Limitations, Cost Sensitivity & Return Trade-off

1. **Proxy Limitations:** ETFs `SOXX` and `XLK` track broad semiconductor and technology sector companies (NVIDIA, Apple, Microsoft, Broadcom, etc.). They contain significant non-AI revenues and are liquid market proxies rather than pure-play AI assets.
2. **Cost Sensitivity:** In backtests, transaction costs of 10 bps (base) are applied per side on rebalancing. Sensitivity analysis across 5 bps and 25 bps confirms that weekly rotation frequency maintains net excess risk-adjusted returns without excessive fee erosion.
3. **Return Trade-off Honesty:** De-risking into short-duration Treasuries (`SHY`) during high-risk regimes cushions severe peak-to-trough drawdowns, but necessarily incurs an opportunity cost (percentage points of return sacrificed during extended bull runs).
